In [1]:
# import some useful libraries
import numpy as np
from scipy import io, signal, fft, interpolate
from scipy.signal import windows, detrend
from matplotlib import pyplot as plt

def mtcsd(x, fs=1, nperseg=256, nfft=None, noverlap=None, nw=3, ntapers=None, detrend_method='constant'):
    """
    Compute the pair-wise cross-spectral density for all channels in an array using Slepian tapers. Adapted from
    the mtcsd function in the labbox Matlab toolbox (authors: Partha Mitra, Ken Harris).
    
    Parameters
    ----------
    x : ndarray
        2D array of signals across which to compute CSD, columns should correspond to channels
    fs : float (default = 1)
        sampling frequency
    nperseg : int, None (default = None)
        number of data points per segment, if None nperseg is set to 256
    nfft : int, None (default = None)
        number of points to include in scipy.fft.fft, if None nfft is set to 2 * nperseg, if nfft > nperseg data 
        will be zero-padded
    noverlap : int, None (default = None)
        amout of overlap between consecutive segments, if None noverlap is set to nperseg / 2
    nw : int (default = 3)
        time-frequency bandwidth for Slepian tapers, passed on to scipy.signal.windows.dpss
    ntapers : int, None (default = None)
        number of tapers, passed on to scipy.signal.windows.dpss, if None ntapers is set to nw * 2 - 1 (as 
        suggested by original authors)
    detrend_method : {'constant', 'linear'} (default = 'constant')
        method used by scipy.signal.detrend to detrend each segment
        
    Returns
    -------
    f : ndarray
        frequency bins
    csd : ndarray
        full cross-spectral density matrix
    """
    
    assert x.ndim == 2, "Data array must be 2D"
    
    # set some default for parameters values     
    if nfft is None:
        nfft = nperseg * 2
    
    if noverlap is None:
        noverlap = nperseg / 2
        
    if ntapers is None:
        ntapers = 2 * nw - 1
        
    stepsize = nperseg - noverlap
    nsegs = int(np.floor(len(x) / stepsize))

    fftnorm = np.sqrt(2) # value taken from original matlab function
    csdnorm = ntapers * nsegs
    
    # initialize csd matrix
    csd = np.zeros((x.shape[1], x.shape[1], nfft), dtype='complex128')
    
    # get FFT frequency bins
    f = fft.fftfreq(nfft, 1/fs)
    
    # get tapers
    tapers = windows.dpss(nperseg, nw, Kmax=ntapers)

    # loop over segments
    for seg_ind in range(nsegs):
        
        # prepare segment
        i0 = int(seg_ind * stepsize)
        i1 = int(seg_ind * stepsize + nperseg)
        if i1 > len(x): # stop if segment is out of range of data
            break
        seg = x[i0:i1, :]
        seg = detrend(seg, type=detrend_method, axis=0)
    
        # apply tapers
        tapered_seg = np.full((len(tapers), seg.shape[0], seg.shape[1]), np.nan)
        for taper_ind, taper in enumerate(tapers):
            tapered_seg[taper_ind] = (seg.T * taper).T    
        
        # compute FFT for each channel-taper combination
        pxx = fft.fft(tapered_seg, n=nfft, axis=1) / fftnorm
        
        # fill upper triangle of csd matrix by averaging values over all tapers and segments
        # normalization factor takes care of the number of tapers and segments
        for ch1 in range(x.shape[-1]): # loop over all unique channel combinations
            for ch2 in range(ch1, x.shape[-1]):
                # compute csd bewteen channels, sum over tapers, normalize
                csd[ch1, ch2, :] += (pxx[:, :, ch1] * np.conjugate(pxx[:, :, ch2])).sum(axis=0) / csdnorm
                
    # fill lower triangle of csd matrix with complex conjugate of upper triangle
    for ch1 in range(x.shape[-1]):
        for ch2 in range(ch1 + 1, x.shape[-1]):
            csd[ch2, ch1, :] = np.conjugate(csd[ch1, ch2, :])
    
    return f, csd



ModuleNotFoundError: No module named 'numpy'

In [2]:
import numpy as np
from scipy import io, signal, interpolate
from matplotlib import pyplot as plt

def butter_filter(data, fs=1, f0=0, f1=7):
    """
    Bandpass filter data between f0 and f1 (Hz) with a butterworth filter, forwards and backwards filters applied.
    
    Parameters
    ----------
    x : ndarray
        1D time-series data
    fs : float
        sampling frequency (Hz)
    f0 : float
        lower limit of frequency range (Hz)
    f1 : float
        upper limit of frequency range (Hz)
    
    Returns
    -------
    out : ndarray
        filtered time-series
    """
    w0 = f0 / (fs / 2) # fraction of Nyquist frequency == 1/2 sampling rate
    w1 = f1 / (fs / 2)
    sos = signal.butter(5, [w0, w1], btype='bandpass', output='sos')
    return signal.sosfiltfilt(sos, data)

def hilbert(x, fs=1, axis=0):
    """
    Use the Hilbert transform/ analytic signal to extract the instantaneous phase, frequency, and amplitude of
    a signal.

    Parameters
    ----------
    x : ndarray
        time-series data
    fs : float (default = 1)
        sampling frequency (Hz)
    axis : int (default=0)
        axis of x along which to compute the analytic signal
        
    Returns
    -------
    phase, frequency, amplitude : ndarray x 3
        arrays of instantaneous phase (in range -pi to pi), frequency, and amplitude
    """
    y = signal.hilbert(x, axis=axis) # complex analytic signal
    phase = np.angle(y) # CCW angle from positive real axis
    # rate of change of phase
    freq = np.gradient(np.unwrap(phase), axis=axis) / (2*np.pi) * fs
    amp = np.abs(y) # length of signal vector at each timepoint
    return phase, freq, amp

def circhist(alpha, n_bins=8, proportion=True, wrap=False):
    """
    Compile a circular histogram.
    
    Parameters
    ----------
    alpha : ndarray
        array of angles, treated as unwrapped array
    n_bins : int
        number of phase bins
    porportion : bool
        if True proportions will be returned rather than counts
    wrap : bool
        if Truen first bin of histogram is repeated at the end (for use with plot_circhist)
        
    Return
    ------
    counts : ndarray
        count or proportion of obervations in each bin
    bins : ndarray
        bin edges
    """
    bins = np.linspace(-np.pi, np.pi, n_bins + 1, endpoint=True)
    weights = np.ones(len(alpha))
    if proportion: 
        weights /= len(alpha)
    counts, bins = np.histogram(alpha, bins=bins, weights=weights)
    if wrap: 
        counts = np.concatenate([counts, [counts[0]]])
        bins = np.concatenate([bins, [bins[0]]])
    return counts, bins

def plot_circhist(alpha, ax=None, max_r=None, n_bins=8, proportion=True):
    """
    Compile and plot a circular histogram and mean resultant vector angle.
    
    Parameters
    ----------
    alpha : ndarray
        array of angles, treated as unwrapped array
    ax : matplotlib.axes._subplots.PolarAxesSubplot
        matplotlib axis on which to plot histogram, if None new axis object will be generated
    mar_r : float
        maximum count/ proportion value to plot, if None will be set to maximum count/ proportion
    n_bins : int
        number of phase bins
    porportion : bool
        if True histogram values will be proportions rather than counts
        
    Return
    ------
    ax : matplotlib.axes._subplots.PolarAxesSubplot
       matplotlib axis object 
    """
    # get phase bins, counts, an circular mean
    counts, bins = circhist(alpha, n_bins=n_bins, proportion=proportion, wrap=True)
    mrl, theta = circmean(alpha)
    
    # set a maximum value
    if max_r is None:
        max_r = counts.max()
    
    # get half bin width
    half_bin = (2 * np.pi) / (len(bins) - 1) / 2
    
    # init polar plot
    if ax is None:
        fig, ax = plt.subplots(subplot_kw={'polar':True})
    
    # plot spike histogram and mean phase
    ax.plot(bins[:-1] + half_bin, counts, lw=2, marker=' ', alpha=.5)
    ax.scatter(theta, max_r)
    
    # format the plot
    ax.scatter(0, 0, fc='none', ec='gray', s=5) # plot center dot
    # outline at max. radius
    ax.plot(np.linspace(-np.pi, np.pi, 1000), np.full(1000, max_r), marker=' ', c='k', lw=0.5)     
    ax.set_frame_on(False) # remove grid lines
    ax.set_xticks([0,  np.pi / 2, np.pi, 3 * np.pi / 2]) # set angle ticks
    ax.set_xticklabels(['0', '\u03C0 / 2', '\u03C0', '-\u03C0 / 2'])
    ax.tick_params(pad=-5)
    if proportion:
        ax.set_yticks(np.arange(0, max_r, 0.1)) # set grid lines
        ax.set_ylim((0, max_r + 0.05))
    ax.set_rlabel_position(90)

    return ax
    
def circmean(alpha, w=None, axis=None):
    """
    Compute mean resultant vector of circular data.
    
    Parameters
    ----------
    alpha : ndarray
        array of angles
    w : ndarray
        array of weights, must be same shape as alpha
    axis : int, None
        axis across which to compute mean
    
    Returns
    -------
    mrl : ndarray
        mean resultant vector length
    theta : ndarray
        mean resultant vector angle
    """
    # weights default to ones
    if w is None:
        w = np.ones_like(alpha)
    w[np.isnan(alpha)] = 0
    
    # compute weighted mean    
    mean_vector = np.nansum(w * np.exp(1j * alpha), axis=axis) / w.sum(axis=axis)
    mrl = np.abs(mean_vector)
    theta = np.angle(mean_vector)
    
    return mrl, theta 


In [4]:
# useful packages
import numpy as np
from scipy import io, signal, fft, interpolate
from scipy.signal import windows, detrend
from matplotlib import pyplot as plt
import scipy

In [8]:
phase

array([-0.93323048, -0.61082922, -0.59324302, ..., -1.5744264 ,
       -1.57408335, -1.57235297])

In [1]:
# load the data
# Data:  16 channels of hippocampal LFP time series from a file lfp_1shank.mat.
# Matrix lfps (1250Hz sampling rate samples x 16 channels) 

lfps = io.loadmat('ws_data_1shank.mat')['lfps'] # array of LFP data
fs = 1250 # sampling frequency
ch = 5 # pick a channel
lfp = lfps[:fs*50, ch]
lfp = lfp[:, np.newaxis] # add empty dimension

# pick a channel 
lfp = lfps[:, 10]

# filter in theta range
lfp_theta = butter_filter(lfp, fs=fs, f0=4, f1=12)

# get Hilbert phase
phase, _, _ = hilbert(lfp_theta)

# extract phases for each spike
# spike_phases = {key: phase[units[key]] for key in units}


NameError: name 'io' is not defined

In [10]:
# import useful toolbox for circular statistics
import pycircstat

# plot phase histograms for each unit
for key in spike_phases:
    ax = plot_circhist(spike_phases[key], n_bins=8)
    mrl, theta = circmean(spike_phases[key])
    p, z = pycircstat.rayleigh(spike_phases[key])
    ax.set_title('unit: %d,  R=%.3f, p=%.3f' % (key, mrl, p))

ModuleNotFoundError: No module named 'pycircstat'

In [ ]:
import numpy as np
from matplotlib import pyplot as plt
from scipy import io, fft, signal
from scipy.signal import windows, detrend

def mtptcoh(pt, ct, fs=1, nperseg=None, nfft=None, noverlap=None, nw=3, ntapers=None, detrend_method='constant',
           nspike_min=4):
    """
    Compute the coherence between a set of point processes and a set of continuous signals. 
    
    Parameters
    ----------
    pt : ndarray
        2D array of point processed, columns should correspond to processes (termed units)
    ct : ndarray
        2D array of continous signals, columns should correspond to channels
    fs : float (default = 1)
        sampling frequency
    nperseg : int, None (default = None)
        number of data points per segment, if None nperseg is set to 256
    nfft : int, None (default = None)
        number of points to include in scipy.fft.fft, if None nfft is set to 2 * nperseg, if nfft > nperseg data 
        will be zero-padded
    noverlap : int, None (default = None)
        amout of overlap between consecutive segments, if None noverlap is set to nperseg / 2
    nw : int (default = 3)
        time-frequency bandwidth for Slepian tapers, passed on to scipy.signal.windows.dpss
    ntapers : int, None (default = None)
        number of tapers, passed on to scipy.signal.windows.dpss, if None ntapers is set to nw * 2 - 1 (as 
        suggested by original authors)
    detrend_method : {'constant', 'linear'} (default = 'constant')
        method used by scipy.signal.detrend to detrend each segment
    nspike_min : int
        smallest number of discrete events (per time segment) allowed for a unit to be considered in the
        computation
        
    Returns
    -------
    f : ndarray
        frequency bins
    Sxx_pt, Sxx_ct : ndarray x 2
        'auto-spectra' of point processes and continuos signals respectively
    Sxy : ndarray
        cross-spectra between every combination of point process and continuous signal channels
    Cxy : ndarray
        coherence magnitude between every combination of point process and continuous signal channels
    phloc : ndarray
        phase locking between every combination of point process and continuous signal channels, phase locking
        index in each frequency bin can be computed by taking the angle of matrix each entry
    """
    # allow 1D inputs
    if pt.ndim == 1:
        pt = pt[:, np.newaxis]
    
    if ct.ndim == 1:
        ct = ct[:, np.newaxis]
        
    assert pt.shape[0] == ct.shape[0], "Signals must be same length."
    
    # set some default for parameters values
    if nperseg is None:
        nperseg = 256
        
    if nfft is None:
        nfft = nperseg * 2
    
    if noverlap is None:
        noverlap = nperseg / 2
        
    if ntapers is None:
        ntapers = 2 * nw - 1
        
    stepsize = nperseg - noverlap
    nsegs = int(np.floor(pt.shape[0] / stepsize))

    fftnorm = np.sqrt(2 / nfft) # value taken from original matlab function
    
    # initialize auto-spectra and cross-spectra arrays
    Sxx_pt = np.zeros((pt.shape[1], nfft), dtype='complex128')
    Sxx_ct = np.zeros((ct.shape[1], nfft), dtype='complex128')
    Sxy = np.zeros((pt.shape[1], ct.shape[1], nfft), dtype='complex128')
    phloc = np.zeros((pt.shape[1], ct.shape[1], nfft), dtype='complex128')
    
    # get FFT frequency bins
    f = fft.fftfreq(nfft, 1 / fs)
    
    # get tapers
    tapers = windows.dpss(nperseg, nw, Kmax=ntapers)

    # keep track of number of segments used for each unit 
    unit_nsegs = np.zeros(pt.shape[1]) 
    # loop over segments
    for seg_ind in range(nsegs):
        
        # prepare segment
        i0 = int(seg_ind * stepsize)
        i1 = int(seg_ind * stepsize + nperseg)
        if i1 > len(pt): # stop if segment is out of range of data
            break
        seg_pt = pt[i0:i1, :]
        seg_ct = ct[i0:i1, :]
        
        # get nspikes
        nspikes = seg_pt.sum(axis=0)
        valid_units = np.where(nspikes >= nspike_min)[0]
        
        # restrict analysis for this segment to units with sufficient spikes
        seg_pt = seg_pt[:, valid_units]
        unit_nsegs[valid_units] += 1
        
        # compute FT of point process data
        pxx_pt = point_fft(seg_pt, tapers, nfft) / fftnorm
        
        # compute 'auto-spectra' of point process
        Sxx_pt[valid_units] += (pxx_pt * np.conjugate(pxx_pt)).sum(axis=0) 
       
        # detrend continuous data
        seg_ct = detrend(seg_ct, type=detrend_method, axis=0)

        # apply tapers
        tapered_seg_ct = np.array([seg_ct.T * taper for taper in tapers]) 

        # compute FFT for each channel-taper combination
        pxx_ct = fft.fft(tapered_seg_ct, nfft) / fftnorm
        
        # compute 'auto-spectra' of continuous process
        Sxx_ct += (pxx_ct * np.conjugate(pxx_ct)).sum(axis=0)
        
        # for each unit-channel combination
        for unit in range(seg_pt.shape[1]):
            for channel in range(seg_ct.shape[1]):
                # compute cross-spectra, averaging over tapers
                Sxy[unit, channel] += (pxx_pt[:, unit, :] * np.conjugate(pxx_ct[:, channel, :])).sum(axis=0)
                # take only phase values and compute PLI, averagins over tapers
                pt_phase = np.angle(pxx_pt[:, unit, :])
                ct_phase = np.angle(pxx_ct[:, channel, :])
                phloc[unit, channel] += (np.exp(1j * pt_phase) * np.conjugate(np.exp(1j * ct_phase))).sum(axis=0)  
                
    # get normalization factors to account for summing over tapers and segments
    unit_csdnorm = unit_nsegs * ntapers
    continuous_csdnorm = nsegs * ntapers
    
    # apply normalization factors (divide by zero will result in NaN)
    Sxx_pt = (Sxx_pt.T / unit_csdnorm).T
    Sxx_ct = (Sxx_ct.T / continuous_csdnorm).T
    Sxy = (Sxy.T / unit_csdnorm).T
    phloc = (phloc.T / unit_csdnorm).T
    
    # compute power normalization matrix
    powernorm = np.zeros((pt.shape[1], ct.shape[1], nfft))
    for unit in range(pt.shape[1]):
        for channel in range(ct.shape[1]):
            powernorm[unit, channel] = np.sqrt((np.abs(Sxx_pt[unit])) * (np.abs(Sxx_ct[channel]))) 
            
    # apply normalization to cross-spectra to get coherence
    Cxy = np.abs(Sxy) / powernorm
    
    return f, Sxx_pt, Sxx_ct, Sxy, Cxy, phloc

def inds2train(inds_list, n):
    """
    Convert a set of event indices (time points) into a binary time-series. It is assumed that all channels in
    the input set occur over the same period and that the indices refer to the same time base.
    
    Parameters
    ----------
    inds_list : list, dict, ndarray
        sequence containing indices (time points) at which events occur for each of a set of 
        point processes
    n : int
        total length of segment in which events occur
    
    Returns
    -------
    event_train : ndarray
        binary time-series array for each member of input set, columns correspond to channels
    """
    event_train = np.zeros((n, len(inds_list)), dtype='uint8')
    for ch in range(len(inds_list)): 
        event_train[inds_list[ch], ch] = 1
    
    return event_train

def point_fft(pt, tapers, nfft):
    """
    Compute Fourier spectra of a set of point processes given a set of tapers.
    
    Parameters
    ----------
    pt : ndarray
        2D array of point processes, columns should correspond to processes
    tapers : ndarray
        2D array of tapers, rows should correspond to different tapers
    nfft : int
        number of points to include in the FFT, if nfft is greater than the number of time points the signals
        will be zero-padded
        
    Returns
    -------
    out : ndarray
        Fourier series of each tapered point process
    """
    # allow 1D input
    if pt.ndim == 1:
        pt = pt[:, np.newaxis]
        
    assert pt.shape[0] = tapers.shape[1], "Tapers and point-processes must have same number of time points."
    
    # apply tapers to point process
    tapered_pt = np.array([pt.T * taper for taper in tapers])
    
    # compute mean event rate (repeated for each taper)
    mean_rates = np.ones((tapers.shape[0], pt.shape[-1])) * pt.mean(axis=0)
    
    # compute FT of tapers
    taper_fft = fft.fft(tapers, nfft)
    
    # compute a 'baseline' spectrum by scaling the taper FT by the mean rate
    baseline = np.array([mean_rates.T * taper for taper in taper_fft.T]).T
    
    # compute FT of tapered point process and subtract baseline  
    return fft.fft(tapered_pt, nfft) - baseline    

In [ ]:
# whitening
def autocovariance(X, p):
    '''
    calculate autocovariance for some data X
    p: time lag
    '''
    scale = len(X) - p
    autoCov = 0
    for i in np.arange(0, len(X)-p):
        autoCov += ((X[i-p]))*(X[i])
        
    return autoCov/scale
    

def stack_cov(X, p):
    '''
    return a autocovariance matrix based on the Yule-Walker equation
    from data X of order p
    '''
    
    Xt_out = []
    cov = [autocovariance(X, i) for i in range(p)]
    cov_reverse = [autocovariance(X, p-i-1) for i in range(p)]
    
    for i in range(p):
        left = cov_reverse[:i]
        right = cov[:p-i]
        Xt_out.extend(left)
        Xt_out.extend(right)
        
    return np.array(Xt_out).reshape(p, p)


def AR(p, X):
    '''
    estimate AR(p) coefficients a for data X by solving the Yule-
    Walker equation in matrix form
    
    p: order of AR model
    '''
    
    X_mat = stack_cov(X, p)
    
    Y = [autocovariance(X, i) for i in range(1, p+1)]
    inv_XX = np.linalg.inv(np.dot(X_mat.T, X_mat))
    XY = np.dot(X_mat, Y)
    a = np.dot(inv_XX, XY)
    
    return a


def predict(a, X):
    '''
    estimate signal with AR coefficients a for data X
    '''
    
    Xt_out = []
    
    for t in range(len(a), len(X)):
        
        temp = 0
        
        for k in range(len(a)):
            temp += a[k]*X[t-k]
        
        Xt_out.append(temp)
        
    return Xt_out

def sigma2(a, x0, x_prev):
    '''
    calculate sigma^2 from Yule-Walker equation
    x0: data at t = 0 from AR estimated signal
    x_prev: data at t = 0-p to t = 0-1 from original signal
    return the noise/innovation/error term at t = 0
    '''
    
    p = len(a)
    variance = 0
    for i, a_i in enumerate(a):
        variance += a_i * x_prev[len(x_prev)-i-1]
    
    sigma = x0 - variance
    
    return sigma

def whiten_signal(signal):
    ar_order = 2
    ar_coeffs = AR(ar_order, signal)
    ar_pred = predict(ar_coeffs, signal)
    return [sigma2(ar_coeffs, ar_pred[i], signal[i:i+ar_order]) for i in range(len(ar_pred))]

In [ ]:
def analytic_signal(signal, filter_from, filter_to):
    signal_whitened = whiten_signal(signal)
    sos = ssignal.butter(2, [filter_from, filter_to], btype='bandpass', fs=fs, output='sos')
    fltp = ssignal.sosfilt(sos, signal_whitened)
    analytic_signal = ssignal.hilbert(fltp)
    amp_env = np.abs(analytic_signal)
    inst_phase = np.angle(analytic_signal)
    inst_freq = (np.diff(inst_phase) / (2.0 * np.pi) * fs)
    return amp_env, inst_phase, inst_freq